# EDA — RSNA 2023 Abdominal Trauma Detection (Grupo G7)

**Disciplina:** Tópicos Especiais em Sistemas de Informação — TP1 (Baseline)
**Grupo:** G7 — Luis Antônio Pereira Rabelo, Francisco Ruan da Silva Queiroz, Carlos Daniel da Silva Ribeiro
**Desafio:** RSNA 2023 Abdominal Trauma Detection

Este notebook cobre o **Marco 1 (Semana 1)** do cronograma:
- leitura de um exame DICOM do início ao fim (metadados, HU, janelamento);
- amostragem estratificada do dataset (documentada e com semente fixa);
- análise exploratória: distribuição de classes, contagem de pacientes/séries, exemplos visuais por classe.

> **Antes de rodar:** faça o cadastro no Kaggle, aceite os termos da competição
> `rsna-2023-abdominal-trauma-detection` e baixe (ou monte via API do Kaggle) os
> arquivos `train.csv`, `train_series_meta.csv` e uma amostra de `train_images/`.
> Ajuste `DATA_DIR` na célula de configuração abaixo — **não use caminho absoluto
> da sua máquina no repositório final**, use uma variável de ambiente ou um caminho
> relativo, conforme o critério de reprodutibilidade do TP1.


In [ ]:
# Dependências
# pip install pydicom SimpleITK pandas numpy matplotlib scikit-image --quiet

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom

%matplotlib inline


## 1. Configuração

Tudo que muda de máquina para máquina fica aqui. Semente fixa para reprodutibilidade
(exigência do §4.4 / §4.5 do enunciado).

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Caminho para a pasta do dataset baixado do Kaggle.
# Recomendado: variável de ambiente, para não hardcodar caminho absoluto pessoal.
DATA_DIR = Path("/kaggle/input/competitions/rsna-2023-abdominal-trauma-detection")

TRAIN_CSV = DATA_DIR / "train_2024.csv"
SERIES_META_CSV = DATA_DIR / "train_series_meta.csv"
IMAGES_DIR = DATA_DIR / "train_images"

# Tamanho da amostra estratificada (número de pacientes) para o baseline.
# Documentar aqui o critério de inclusão: N pacientes por classe de gravidade
# do órgão de maior severidade (ver Seção 2 abaixo).
N_PATIENTS_SAMPLE = 180

print("DATA_DIR:", DATA_DIR.resolve())


## 2. Rótulos e metadados de série

O dataset tem rótulos em nível de **paciente** (`train.csv`) para 5 alvos:
`bowel`, `extravasation`, `kidney`, `liver`, `spleen` — cada um com níveis de
gravidade próprios (a maioria binária `healthy`/`injury`, exceto kidney/liver/spleen
que têm `healthy`/`low`/`high`), além de um alvo agregado `any_injury`.

`train_series_meta.csv` traz, por paciente/série, se há trauma contuso ou
penetrante e a contagem aproximada de cortes.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
series_meta_df = pd.read_csv(SERIES_META_CSV)

print("train.csv:", train_df.shape)
print("train_series_meta.csv:", series_meta_df.shape)

train_df.head()


In [ ]:
series_meta_df.head()


## 3. Distribuição de classes por órgão

Ponto crítico deste desafio (conforme o enunciado): **múltiplos órgãos e múltiplas
gravidades simultâneas**. Olhar cada alvo isoladamente primeiro.

In [ ]:
organ_targets = {
    "bowel": ["bowel_healthy", "bowel_injury"],
    "extravasation": ["extravasation_healthy", "extravasation_injury"],
    "kidney": ["kidney_healthy", "kidney_low", "kidney_high"],
    "liver": ["liver_healthy", "liver_low", "liver_high"],
    "spleen": ["spleen_healthy", "spleen_low", "spleen_high"],
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (organ, cols) in zip(axes, organ_targets.items()):
    present_cols = [c for c in cols if c in train_df.columns]
    counts = train_df[present_cols].sum().sort_values(ascending=False)
    counts.plot(kind="bar", ax=ax, color="#4C72B0")
    ax.set_title(organ)
    ax.set_ylabel("nº de pacientes")
    ax.tick_params(axis="x", rotation=30)

# alvo agregado any_injury, se existir
if "any_injury" in train_df.columns:
    ax = axes[5]
    train_df["any_injury"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#DD8452")
    ax.set_title("any_injury (agregado)")
    ax.set_xticklabels(["sem lesão", "com lesão"], rotation=0)

plt.tight_layout()
plt.savefig("class_distribution_by_organ.png", dpi=150)
plt.show()


In [ ]:
# Desbalanceamento em números — importante para justificar a métrica
# e o tratamento de desbalanceamento na Seção 4.3 do artigo.
summary_rows = []
for organ, cols in organ_targets.items():
    present_cols = [c for c in cols if c in train_df.columns]
    counts = train_df[present_cols].sum()
    total = counts.sum()
    for col, n in counts.items():
        summary_rows.append({
            "orgao": organ,
            "classe": col,
            "n_pacientes": int(n),
            "pct": round(100 * n / total, 2) if total else 0.0,
        })

class_summary_df = pd.DataFrame(summary_rows)
class_summary_df.to_csv("class_distribution_summary.csv", index=False)
class_summary_df


## 4. Amostragem estratificada

Amostrar não é problema — amostrar sem documentar, é (enunciado, Seção 2).
Aqui: amostra por paciente, estratificada por `any_injury` (ou, na ausência dele,
por uma combinação simples dos alvos binários), com semente fixa.

In [ ]:
strat_col = "any_injury" if "any_injury" in train_df.columns else organ_targets["bowel"][1]

sampled_parts = []
for value, group in train_df.groupby(strat_col):
    n = min(len(group), max(1, round(N_PATIENTS_SAMPLE * len(group) / len(train_df))))
    sampled_parts.append(group.sample(n=n, random_state=SEED))

sample_df = pd.concat(sampled_parts).reset_index(drop=True)

print(f"Amostra: {len(sample_df)} pacientes de {len(train_df)} "
      f"({100*len(sample_df)/len(train_df):.1f}%)")
print("Critério: estratificação por", strat_col, "| semente:", SEED)

sample_df.to_csv("sample_patient_ids.csv", index=False)
sample_df[strat_col].value_counts()


## 5. Leitura de um exame DICOM do início ao fim

Objetivo: confirmar que sabemos ler a série completa e extrair os metadados
clinicamente relevantes — `RescaleSlope`/`RescaleIntercept` (para converter para
Hounsfield Units), `PixelSpacing`, `SliceThickness`, `WindowCenter`/`WindowWidth`.
Ignorar o rescale é um dos "erros mais comuns" listados no enunciado (Seção 10.6).

In [ ]:
def load_series_slices(patient_id, series_id, images_dir=IMAGES_DIR):
    series_dir = images_dir / str(patient_id) / str(series_id)
    dcm_paths = sorted(series_dir.glob("*.dcm"),
                        key=lambda p: int(p.stem))
    slices = [pydicom.dcmread(p) for p in dcm_paths]
    return slices


def dicom_to_hu(dcm):
    """Converte pixel_array bruto para Hounsfield Units usando rescale slope/intercept."""
    slope = float(getattr(dcm, "RescaleSlope", 1.0))
    intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
    return dcm.pixel_array.astype(np.float32) * slope + intercept


def apply_window(hu_img, window_center, window_width):
    """Janelamento: recorta e normaliza HU para o intervalo [0, 1] em uma janela clínica."""
    low = window_center - window_width / 2
    high = window_center + window_width / 2
    windowed = np.clip(hu_img, low, high)
    return (windowed - low) / (high - low)


# Exemplo: pegue um patient_id/series_id existente da amostra + série meta
example_patient = sample_df["patient_id"].iloc[0]
example_series = series_meta_df.loc[
    series_meta_df["patient_id"] == example_patient, "series_id"
].iloc[0]

print("Paciente de exemplo:", example_patient, "| Série:", example_series)

slices = load_series_slices(example_patient, example_series)
print("Número de cortes na série:", len(slices))

first = slices[0]
print("RescaleSlope:", getattr(first, "RescaleSlope", None))
print("RescaleIntercept:", getattr(first, "RescaleIntercept", None))
print("PixelSpacing:", getattr(first, "PixelSpacing", None))
print("SliceThickness:", getattr(first, "SliceThickness", None))


In [ ]:
# Janela abdominal padrão (tecidos moles): center ~40 HU, width ~400 HU.
# Ajustar/justificar no artigo com base na literatura de trauma abdominal por TC.
WINDOW_CENTER, WINDOW_WIDTH = 40, 400

mid_idx = len(slices) // 2
hu_img = dicom_to_hu(slices[mid_idx])
windowed_img = apply_window(hu_img, WINDOW_CENTER, WINDOW_WIDTH)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(hu_img, cmap="gray")
axes[0].set_title("HU bruto (sem janela)")
axes[1].imshow(windowed_img, cmap="gray")
axes[1].set_title(f"Janelado (C={WINDOW_CENTER}, W={WINDOW_WIDTH})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("example_windowing.png", dpi=150)
plt.show()


## 6. Exemplos visuais por classe

Uma linha por classe de `any_injury` (ou pelo alvo escolhido), mostrando o corte
central da série — ponto de partida para a análise de erro qualitativa mais
adiante no projeto.

In [ ]:
def show_grid_by_class(df, meta_df, target_col, n_per_class=3, images_dir=IMAGES_DIR,
                        window_center=WINDOW_CENTER, window_width=WINDOW_WIDTH):
    classes = sorted(df[target_col].unique())
    fig, axes = plt.subplots(len(classes), n_per_class,
                              figsize=(3 * n_per_class, 3 * len(classes)))
    axes = np.atleast_2d(axes)

    for row, cls in enumerate(classes):
        subset = df[df[target_col] == cls].sample(
            n=min(n_per_class, (df[target_col] == cls).sum()), random_state=SEED
        )
        for col, (_, patient_row) in enumerate(subset.iterrows()):
            pid = patient_row["patient_id"]
            sid = meta_df.loc[meta_df["patient_id"] == pid, "series_id"].iloc[0]
            s = load_series_slices(pid, sid, images_dir)
            mid = s[len(s) // 2]
            img = apply_window(dicom_to_hu(mid), window_center, window_width)
            axes[row, col].imshow(img, cmap="gray")
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(f"classe={cls}", fontsize=10)
        for col in range(len(subset), n_per_class):
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.savefig("examples_by_class.png", dpi=150)
    plt.show()


show_grid_by_class(sample_df, series_meta_df, strat_col)


## 7. Próximos passos (fora do escopo deste notebook)

- Congelar a partição treino/validação **por paciente** (não por corte/imagem).
- Implementar o pipeline de pré-processamento completo (recorte de ROI, remoção
  de mesa/marcadores, reamostragem).
- Definir a estratégia de agregação corte → exame (2D com pooling, 2.5D, ou 3D),
  já que este é o alvo mais volumoso do desafio.
- Extrair as três famílias de características (Seção 4.2 do enunciado) e treinar
  o baseline trivial + modelos clássicos (Seção 4.3).

Este notebook cobre o **Marco 1**: repositório + EDA. Guarde `class_distribution_summary.csv`,
`sample_patient_ids.csv` e as figuras `.png` geradas — elas alimentam diretamente
a Seção de Metodologia e a análise exploratória do artigo.